# U-Net-2048 proposals + instance refiner on Colab (5 folds + full, pooled OOF)

**Runtime:** GPU (A100 if available, else L4), **High-RAM** on. Then *Runtime > Run all*.

**Kaggle token**, one of:
* Colab secret `KAGGLE_API_TOKEN` (new-style token), or secrets `KAGGLE_USERNAME` + `KAGGLE_KEY`; notebook access on; or
* `kaggle.json` in the root of your Google Drive.

Everything is written to `MyDrive/filament_fusion/`. If the runtime disconnects, reconnect and *Run all*
again: training resumes from the last epoch, finished steps are skipped.

In [ ]:
# 1. Drive, Kaggle token, packages
import os, sys, json, subprocess, time, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/filament_fusion')
for d in ['refiner', 'logs', 'results']:
    (DRIVE / d).mkdir(parents=True, exist_ok=True)
from google.colab import userdata

def secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None

if secret('KAGGLE_API_TOKEN'):  # new-style token (Kaggle settings > API > Generate New Token)
    os.environ['KAGGLE_API_TOKEN'] = secret('KAGGLE_API_TOKEN')
elif secret('KAGGLE_USERNAME') and secret('KAGGLE_KEY'):  # legacy kaggle.json values
    os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = secret('KAGGLE_USERNAME'), secret('KAGGLE_KEY')
else:
    tok = Path('/content/drive/MyDrive/kaggle.json')
    assert tok.exists(), 'add a KAGGLE_API_TOKEN secret (or KAGGLE_USERNAME + KAGGLE_KEY), or put kaggle.json in MyDrive'
    Path('~/.kaggle').expanduser().mkdir(exist_ok=True)
    shutil.copy(tok, Path('~/.kaggle/kaggle.json').expanduser())
    os.chmod(Path('~/.kaggle/kaggle.json').expanduser(), 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle>=1.8', 'segmentation-models-pytorch', 'timm',
                'pycocotools'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil
RAM_GB = psutil.virtual_memory().total / 2**30
print(f'RAM {RAM_GB:.0f} GB, CPUs {os.cpu_count()}')

In [ ]:
# 2. Code (Azumah/ from main, unchanged) and data pack from Kaggle
REPO = Path('/content/aml_project')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'charan/fusion-self',
                    'https://github.com/hsfcoulibaly/aml_project.git', str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
CODE = REPO / 'Azumah'
WORK = Path('/content/fseg')
if not (WORK / 'manifest.csv').exists():
    !kaggle kernels output charanpeeriga/filament-fusion-pack -p /content/pack --file-pattern 'fusion_pack.tar'
    !tar -xf /content/pack/fusion_pack.tar -C /content && rm /content/pack/fusion_pack.tar
if not (WORK / 'refiner').is_symlink():
    shutil.rmtree(WORK / 'refiner', ignore_errors=True)
    (WORK / 'refiner').symlink_to(DRIVE / 'refiner')
os.environ.update(FSEG_WORK_DIR=str(WORK), FSEG_DATA_ROOT='/content/no_raw_data_needed',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
os.chdir(CODE)
print('maps', len(list((WORK / 'unet/maps/unet2048/oof').glob('*.png'))), len(list((WORK / 'unet/maps/unet2048/test').glob('*.png'))),
      '| proposals', sorted(p.name for p in (WORK / 'runs').glob('*/*.pkl')))

In [ ]:
# 3. Train 6 refiners (60 epochs; folds 0-4 held out + full data). Two at a time with High-RAM.
def name(k):
    return 'u2r_full' if k == 'none' else f'u2r_v{k}'

def done(k):
    h = DRIVE / 'refiner' / name(k) / 'history.csv'
    return h.exists() and len(h.read_text().strip().splitlines()) - 1 >= 60

def cmd(k):
    return (f'python scripts/train_refiner.py --name {name(k)} --val-fold {k} --unet-name unet2048 --arch unet '
            f'--epochs 60 --workers 4 --max-matched 12 --max-neg 8 --resume >> {DRIVE}/logs/{name(k)}.log 2>&1')

todo = [k for k in ['0', '1', '2', '3', '4', 'none'] if not done(k)]
parallel = 2 if RAM_GB > 40 else 1
print('to train:', todo, '| parallel', parallel)
T0 = time.time()
running = []
while todo or running:
    while todo and len(running) < parallel:
        k = todo.pop(0)
        running.append((k, subprocess.Popen(cmd(k), shell=True)))
    time.sleep(120)
    for k, p in list(running):
        if p.poll() is not None:
            running.remove((k, p))
            if p.returncode:
                print(f'{name(k)} FAILED, see logs/{name(k)}.log'); print(open(f'{DRIVE}/logs/{name(k)}.log').read()[-3000:])
    for k, _ in running:
        lines = open(f'{DRIVE}/logs/{name(k)}.log').read().strip().splitlines()
        if lines:
            print(f'[{(time.time() - T0) / 60:5.0f} min] {name(k)}: {lines[-1][:160]}', flush=True)
print('all refiners trained:', all(done(k) for k in ['0', '1', '2', '3', '4', 'none']))

In [ ]:
# 4. Out-of-fold refinement of all 707 images (each fold by the refiner that never saw it), pooled PQ
for k in range(5):
    if not (DRIVE / 'refiner' / f'u2r_f{k}' / f'refined_fold{k}.pkl').exists():
        !python scripts/predict_refiner.py oof --fold {k} --refiner u2r_v{k} --tag u2r_f{k} --unet-name unet2048 --tta d4 > {DRIVE}/logs/oof_f{k}.log 2>&1
    print(k, open(f'{DRIVE}/logs/oof_f{k}.log').read().strip().splitlines()[-1][:250])
!python scripts/predict_refiner.py pool --tag u2r --folds 0 1 2 3 4 --tta d4 2>&1 | tee {DRIVE}/logs/pool.log

In [ ]:
# 5. Test: 6-refiner vote with the pooled config, and the full-data refiner alone -> Drive
P = WORK / 'refiner' / 'u2r_pooled'
cfg = P / 'pq_config.json'
raw = WORK / 'runs' / 'unet2048_test' / 'test_raw.pkl'
six = ' '.join([f'u2r_v{k}' for k in range(5)] + ['u2r_full'])
R = DRIVE / 'results'
!python scripts/predict_refiner.py test --refiner {six} --config {cfg} --raw {raw} --unet-name unet2048 --tag test_u2rx6 --out {R}/submission_u2rx6_pooled.csv 2>&1 | tail -2
!python scripts/predict_refiner.py test --refiner u2r_full --config {cfg} --raw {raw} --unet-name unet2048 --tag test_u2rfull --out {R}/submission_u2rfull_pooled.csv 2>&1 | tail -2
shutil.copy(cfg, R / 'pq_config_pooled.json')
shutil.copy(P / 'per_fold.csv', R / 'per_fold.csv')
shutil.copy(P / 'pq_sweep_pooled.csv', R / 'pq_sweep_pooled.csv')
print(open(cfg).read()); print(open(P / 'per_fold.csv').read())
!ls -la {R}